# GOAL: build naive model

Main idea: prediction on a particular day = average of all readings on that day before. 

If there are no readings on the same day before, do forward fill.


In [1]:
#import everything!
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from copy import deepcopy
from clean_and_collect_power_data import Clean
from PreRun import PreRun
from sklearn.metrics import mean_absolute_error, mean_squared_error

#path to data
read_path = "../../../../data_ds_project/parquet_cleaned_energy"
#systems
good_systems_list = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]
reader_types = ["meter", "inverter", None]
#systems_cleaned
systems_cleaned = pd.read_csv("../../../data/core/systems_cleaned.csv")


In [2]:
def naive_energy_forecaster(past_data: pd.DataFrame, times_to_predict: pd.DataFrame):
    df = past_data.copy()
    df['month_day'] = df['time'].dt.strftime('%m-%d')
    df['time_of_day'] = df['time'].dt.time

    avg_energy = (
        df.groupby(['month_day', 'time_of_day'])['energy']
        .mean()
        .reset_index(name='energy_pred')
    )

    predictions = times_to_predict.copy()
    predictions['month_day'] = predictions['time'].dt.strftime('%m-%d')
    predictions['time_of_day'] = predictions['time'].dt.time
    predictions = predictions.merge(
        avg_energy,
        on=['month_day', 'time_of_day'],
        how='left'
    )

    predictions['energy_pred'] = predictions['energy_pred'].ffill().bfill()
    
    return predictions['energy_pred']

In [3]:
#custom error

def custom_error(y_true, y_pred, a=1, b=2):
    """custom error to penalize overestimation. Like a weighted MSE.

    Args:
        y_true (_type_): _description_
        y_pred (_type_): _description_
        a (int, optional): coefficient of underestimation. Defaults to 1.
        b (int, optional): coefficient of overestimation. Defaults to 1.

    Returns:
        float: mean error
    """
    if a<0 or b<0:
        raise ValueError("a and b must be non-negative")
    elif a==0 and b==0:
        raise ValueError("a and b cannot both be zero")


    Y = pd.DataFrame({'y_true': y_true.reset_index(drop=True), 'y_pred': y_pred.reset_index(drop=True)})
    
    Y['error'] = np.where(Y['y_true'] > Y['y_pred'], a * (Y['y_true'] - Y['y_pred'])**2, b * (Y['y_pred'] - Y['y_true'])**2)
    
    return Y['error'].mean()
    

In [4]:
# experiment with system 4
system_id=4
check_prerun = PreRun(system_id=system_id, meter_or_inverter=None, path=read_path, systems_cleaned=systems_cleaned)
check_prerun.fill_missing_hours()
#do train test split
#print("Good days:", check_prerun.good_days)
good_days = check_prerun.good_days['date'].dt.date
train_days = good_days[:int(0.8*len(good_days))]
test_days = good_days[int(0.8*len(good_days)):]
set_train_days = set(train_days)
set_test_days = set(test_days)

#print("check_prerun.data", check_prerun.data.head())

train_data = check_prerun.data[check_prerun.data['time'].dt.date.isin(set_train_days)]
test_data = check_prerun.data[check_prerun.data['time'].dt.date.isin(set_test_days)]

# print("Train data:")
# print(train_data.head())

y_pred = naive_energy_forecaster(train_data, pd.DataFrame(test_data['time']))
y_true = test_data['energy']

#print(type(y_true.iloc[0]), type(y_pred.iloc[0]))



print("MAE:", mean_absolute_error(y_true, y_pred))
print("MSE:", mean_squared_error(y_true, y_pred))
print("custom error", custom_error(y_true, y_pred, 1,2))


MAE: 0.1115848676929585
MSE: 0.026589312238846266
custom error 0.044728375998536365
